# Improve a scheduler

Jobs of different sizes arrive at a group of workers. Where should each job go?
Our starting program takes turns: worker 0, worker 1, and so on. That is cheap,
but when large jobs arrive at regular intervals, the same workers get overloaded.

Try a small change: **look at two workers and choose the less busy one.**
Here is the proposed function. `load()` asks for a worker's current load;
each request counts as one **probe**:

```py
def choose(cluster, size):
    first = cluster.pick()
    second = cluster.pick()
    if second == first:
        second = (first + 1) % cluster.size
    return first if cluster.load(first) <= cluster.load(second) else second
```

The fixed simulator actually runs this Python code. Its development workloads
produce this comparison:

| Program | Load imbalance ↓ | Probes per job ↓ | Quality ↑ |
|---|---:|---:|---:|
| Starting round-robin | 0.2071 | 0 | 0.7929 |
| Inspect every worker | 0.0381 | 7 | 0.8219 |
| Inspect two workers | 0.0464 | 2 | 0.9136 |

**Two probes buy much better balance than the starting program.** Inspecting
every worker balances jobs a little better, but takes seven probes per job here.
The two-worker program offers the better tradeoff under the fixed score:
`quality = max(0, 1 - imbalance - 0.02 * probes_per_job)`.
Imbalance is zero for an even distribution and one when a single worker gets
everything. These measurements describe this simulator, not hardware speed.

The proposer tries handwritten revisions, including mistakes. The measured
scores come from running those programs, not a lookup table or a live model.

## Run it in Jupyter

This is an optional next step after [your first improvement](https://sentient-xyz.github.io/meta-evolve-docs/start-here/).
Use a **fresh Jupyter kernel with Python 3.12 or newer**, on macOS with usable
`sandbox-exec`, or Linux with usable **Landlock ABI 3+**. The runtime must allow
confined child processes. The third cell checks that they really work before
the study starts. Hosted notebooks, including Colab, are not verified here.

Save the notebook in an ordinary project folder, outside your Python installation
and, on Linux, outside `/tmp`, `/var/tmp`, `/usr`, and other system directories.
Its downloaded code needs a protected location; temporary execution workspaces
are created separately. No model key, GPU, or repository checkout is needed.



Run these four cells in order. Installation and download use the same published
documentation build. An archived copy also works: put its `meta-evolve.zip` and
`systems-example.zip` beside the notebook; the cells find them automatically.
Source installation may download build tools. If you have already imported
Meta-Evolve or the example, restart the kernel before starting again.

### 1. Install

In [ ]:
from pathlib import Path

downloads = "https://sentient-xyz.github.io/meta-evolve-docs/downloads/"
package = "./meta-evolve.zip" if Path("meta-evolve.zip").is_file() else downloads + "meta-evolve.zip"
%pip install {package}

### 2. Download the complete study

In [ ]:
import sys
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

archive = Path("systems-example.zip")
if archive.is_file():
    study_bytes = archive.read_bytes()
else:
    with urlopen(downloads + "systems-example.zip", timeout=30) as response:
        study_bytes = response.read()
study_root = (Path.cwd() / "scheduler-example").resolve()
with ZipFile(BytesIO(study_bytes)) as bundle:
    bundle.extractall(study_root)
sys.path.insert(0, str(study_root))
from examples.research.systems_optimization import command, confinement

print("Complete study downloaded and imported.")

### 3. Check this runtime

This probe starts a confined Python process and checks that it cannot read
the downloaded workload file. It is setup, outside the study's measured work.
If it fails, the following cells leave the study stopped and explain the remedy.

In [ ]:
import subprocess
from tempfile import TemporaryDirectory

runtime_ready = False
protected = study_root / "examples/research/systems_optimization/workloads.py"
probe = """import errno, pathlib, sys
try:
    pathlib.Path(sys.argv[1]).read_bytes()
except PermissionError as error:
    if error.errno not in (errno.EACCES, errno.EPERM):
        raise
    print('protected')
else:
    raise SystemExit('Workloads were readable: confinement did not hold.')
"""
try:
    protected.read_bytes()
    if not Path(command.__file__).resolve().is_relative_to(study_root):
        raise RuntimeError("An older example is imported; restart the kernel.")
    with TemporaryDirectory(prefix="scheduler-check-") as workspace:
        checked = subprocess.run(
            [*confinement.launcher(Path(workspace)), sys.executable,
             "-I", "-B", "-c", probe, str(protected)],
            cwd=workspace, capture_output=True, text=True, timeout=10,
        )
    if checked.returncode or checked.stdout.strip() != "protected":
        raise RuntimeError((checked.stderr or checked.stdout).strip() or "Probe failed.")
    runtime_ready = True
    print(f"Ready: Python {sys.version.split()[0]}, {confinement.mechanism()} enforced.")
except (OSError, RuntimeError, subprocess.TimeoutExpired) as error:
    print(f"Study stopped: {error}")
    print("Use local Jupyter on macOS or Linux with the confinement listed above.")
    print("Keep the notebook outside system/temp folders and the Python installation.")

### 4. Run and compare

The study tests the starting program and eight revisions, then selects a program
on a separate workload and checks it once more on a final workload. This cell
shows the development comparison from its recorded measurements.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

def show_measurements(attempt, label):
    if attempt is None:
        print(f"{label}: no measured candidate")
        return
    if not attempt.rankable:
        print(f"{label}: {attempt.failure_kind}/{attempt.outcome}")
        return
    metrics = attempt.metrics
    print(f"{label:24} {metrics['imbalance']:9.4f}"
          f" {metrics['work']:7.1f} {metrics['quality']:9.4f}")

result = None
selected_source = None
if runtime_ready:
    with TemporaryDirectory(prefix="scheduler-run-") as output:
        result = command.run_example(Path(output))
        chosen = result.decision.selection.chosen
        if chosen is not None:
            trial = next(item for item in result.search.trials()
                         if item.logical_step == chosen.logical_step)
            selected_source = trial.artifact.value["scheduler.py"]
    development = {item.candidate: item for item in result.decision.development}
    print("Development workload       Imbalance  Probes   Quality")
    for name, label in (
        ("round-robin (seed)", "Starting round-robin"),
        ("least-loaded-scan", "Inspect every worker"),
        ("power-of-two-choices", "Inspect two workers"),
    ):
        show_measurements(development.get(name), label)
else:
    print("Study not started. Resolve the runtime check above first.")

Rerunning this cell starts a fresh study. Its temporary store is cleaned up even
if execution raises; the comparison and selected source remain inspectable below.

## Check the selected program

Development feedback helps propose changes. **Selection** tests eligible programs
on a different, published workload after proposing ends. **Final checking** tests
only the starting and selected programs after selection. Keeping those phases
separate prevents treating a development score as a result on new jobs.

In [ ]:
if result is not None:
    decision = result.decision
    print("Selected on the selection workload:", decision.selection.winner)
    comparison = decision.comparison
    if comparison is None:
        print("No program was selected; no final comparison was made.")
    else:
        print("Final workload             Imbalance  Probes   Quality")
        show_measurements(comparison.baseline, "Starting round-robin")
        show_measurements(comparison.selected, "Selected program")
        print("Final verdict:", comparison.verdict)
    if selected_source is not None:
        print("\nSelected scheduler.py:\n" + selected_source)
else:
    print("Run the study after the runtime check passes.")

The selected program is `power-of-two-choices`. On the final workload, imbalance
falls from **0.2088 to 0.0794**, using **two probes instead of zero**; quality
rises from **0.7912 to 0.8806**. These results do not establish production gains.
The source above is the actual selected artifact. Selection does not deploy it.

## Change one line

What if the comparison used `>=` instead of `<=`, choosing the *busier* worker?
Change the selected source and run it through the same fixed development
evaluator. This optional check runs separately from the recorded study above.

In [ ]:
from examples.research.systems_optimization import execution, surface

if selected_source is not None and "<=" in selected_source:
    edited_source = selected_source.replace("<=", ">=")
    measured = execution.evaluate(surface.with_scheduler(edited_source), "development")
    if measured.failure is not None:
        print("Edit rejected:", measured.failure.kind, measured.failure.message)
    else:
        print("Edited program on the development workload:")
        for metric in ("imbalance", "work", "quality"):
            print(f"{metric}: {measured.metrics[metric]:.4f}")
else:
    print("This exercise needs a selected program with the <= comparison.")

Both comparisons use two probes. Reversing the comparison lowers quality from
**0.9136 to 0.7731**. Change the replacement back to `"<="` and rerun this cell:
quality returns to **0.9136**. You can also set `edited_source = "not valid Python!"`
to see a typed rejection. Keep the evaluator fixed as you change the program.

## Advanced: inspect the governed study

In [ ]:
if result is not None:
    print("Rejected revisions from the original study (no score):")
    for attempt in result.decision.development:
        if not attempt.rankable:
            print(f"{attempt.candidate}: {attempt.failure_kind}/{attempt.outcome}")
    print("Evaluations by phase:", dict(result.decision.accounting.evaluations))
    print("Scenario runs by phase:", dict(result.decision.accounting.scenario_runs))
    print("All replay checks:", all(result.replayed.values()))
    print("Decision digest:", result.decision_digest)
else:
    print("Run the study after the runtime check passes.")

These counts cover the original study, before the optional edit check.
`private` names selection; `held_out` names final checking. Replay verifies the
retained history without rerunning any scheduler. To retain the store, run this
command from `scheduler-example/` with a new output directory:

```sh
python examples/research/systems_optimization/main.py --output saved-study
```

The [complete study source](https://github.com/sentient-xyz/meta-evolve/tree/main/examples/research/systems_optimization)
explains identities, accounting, and the file-access boundary. Confinement does
not claim general network or hostile-process isolation.

<a id="research-question"></a>

### Research question: ADRS

Can verifier-guided revisions improve a systems algorithm? This shipped,
fixture-only study is inspired by [ADRS](https://arxiv.org/abs/2512.14806).
It is not a reproduction of ADRS, its ten case studies, or live-model gains.
Use the [paper-reading worksheet](https://sentient-xyz.github.io/meta-evolve-docs/research/decomposition/#the-worksheet) to map the
research mechanism onto the components below.

## Component mapping

| Role | Maintained study |
|---|---|
| Artifact / seed | `SourceTree` containing the round-robin `scheduler.py` |
| Proposer | Eight authored revisions, produced in confined child processes |
| Evaluator / objective | Fixed simulator; maximize quality from imbalance and probes |
| Search | `Greedy` with eight proposal attempts after the seed |
| Feedback | Development observations and verifier failures |
| External controls | Workload splits, confinement, budgets, selection, final checking |
| Retained results | Durable development history plus later measurements and decision record |

<a id="capabilities-dependencies-and-evidence"></a>

Scope and acceptance remain in [#195 B2](https://github.com/sentient-xyz/meta-evolve/issues/195)
and the [milestone record](https://sentient-xyz.github.io/meta-evolve-docs/milestones/#issue-195-b2-systems-optimization-example).
Source checked on 2026-09-15: [primary abstract/report](https://arxiv.org/abs/2512.14806).
Local evidence establishes this fixture's mechanism, not the upstream results.